In [1]:
!nvidia-smi


Sun Nov 15 15:51:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
from collections import Counter

racine = "/kaggle/input"
for dataset in os.listdir(racine):
    print("=== ", dataset)
    ext = Counter()
    for dp, dn, fn in os.walk(os.path.join(racine, dataset)):
        for f in fn:
            ext[os.path.splitext(f)[1].lower()] += 1
    print("extensions :", dict(ext))

===  notebooks
extensions : {'.patch': 7, '.ipynb': 1, '.json': 1, '.html': 1, '.avi': 3, '.css': 1, '.csv': 2, '.png': 25, '.py': 3, '.pyc': 3}


In [1]:
import os

for item in os.listdir("/kaggle/input"):
    print(item)

competitions


In [10]:
import os

BASE_DIR = "/kaggle/input/competitions"

print(os.listdir(BASE_DIR))

['deepfake-detection-challenge']


In [11]:
for root, dirs, files in os.walk(BASE_DIR):
    print("\nDossier :", root)
    print("Fichiers :", files[:10])
    print("Nombre :", len(files))


Dossier : /kaggle/input/competitions
Fichiers : []
Nombre : 0

Dossier : /kaggle/input/competitions/deepfake-detection-challenge
Fichiers : ['sample_submission.csv']
Nombre : 1

Dossier : /kaggle/input/competitions/deepfake-detection-challenge/test_videos
Fichiers : ['qooxnxqqjb.mp4', 'hierggamuo.mp4', 'ocgdbrgmtq.mp4', 'icbsahlivv.mp4', 'prhmixykhr.mp4', 'ihglzxzroo.mp4', 'yarpxfqejd.mp4', 'orixbcfvdz.mp4', 'qxyrtwozyw.mp4', 'hqzwudvhih.mp4']
Nombre : 400

Dossier : /kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos
Fichiers : ['eivxffliio.mp4', 'dwediigjit.mp4', 'asvcrfdpnq.mp4', 'dntkzzzcdh.mp4', 'dboxtiehng.mp4', 'elginszwtk.mp4', 'cwbacdwrzo.mp4', 'afoovlsmtx.mp4', 'cmxcfkrjiv.mp4', 'aczrgyricp.mp4']
Nombre : 401


In [12]:
import os
import json

BASE_DIR = "/kaggle/input/competitions/deepfake-detection-challenge"
TRAIN_DIR = os.path.join(BASE_DIR, "train_sample_videos")

print(os.listdir(TRAIN_DIR)[-10:])

['adohikbdaz.mp4', 'ehbnclaukr.mp4', 'czfunozvwp.mp4', 'djvutyvaio.mp4', 'ensyyivobf.mp4', 'bkmdzhfzfh.mp4', 'amaivqofda.mp4', 'cdyakrxkia.mp4', 'cksanfsjhc.mp4', 'drsakwyvqv.mp4']


In [13]:
metadata_path = os.path.join(TRAIN_DIR, "metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("Nombre de vidéos :", len(metadata))

from collections import Counter

labels = Counter(info["label"] for info in metadata.values())

print("Répartition des labels :", labels)

for filename, info in list(metadata.items())[:5]:
    print(filename, info)

Nombre de vidéos : 400
Répartition des labels : Counter({'FAKE': 323, 'REAL': 77})
aagfhgtpmv.mp4 {'label': 'FAKE', 'split': 'train', 'original': 'vudstovrck.mp4'}
aapnvogymq.mp4 {'label': 'FAKE', 'split': 'train', 'original': 'jdubbvfswz.mp4'}
abarnvbtwb.mp4 {'label': 'REAL', 'split': 'train', 'original': None}
abofeumbvv.mp4 {'label': 'FAKE', 'split': 'train', 'original': 'atvmxvwyns.mp4'}
abqwwspghj.mp4 {'label': 'FAKE', 'split': 'train', 'original': 'qzimuostzz.mp4'}


In [1]:
import glob, os
import pandas as pd

racine = "/kaggle/input"
meta = glob.glob(racine + "/**/metadata.json", recursive=True)
print("metadata trouvés :", meta)

dossier = os.path.dirname(meta[0])
videos = glob.glob(dossier + "/*.mp4")
print("dossier :", dossier)
print("nombre de vidéos .mp4 :", len(videos))

df = pd.read_json(meta[0]).T
print("\nlignes, colonnes :", df.shape)
print("colonnes :", df.columns.tolist())
print(df.head())
print("\nrépartition des labels :")
print(df["label"].value_counts())

metadata trouvés : ['/kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos/metadata.json']
dossier : /kaggle/input/competitions/deepfake-detection-challenge/train_sample_videos
nombre de vidéos .mp4 : 400

lignes, colonnes : (400, 3)
colonnes : ['label', 'split', 'original']
               label  split        original
aagfhgtpmv.mp4  FAKE  train  vudstovrck.mp4
aapnvogymq.mp4  FAKE  train  jdubbvfswz.mp4
abarnvbtwb.mp4  REAL  train            None
abofeumbvv.mp4  FAKE  train  atvmxvwyns.mp4
abqwwspghj.mp4  FAKE  train  qzimuostzz.mp4

répartition des labels :
label
FAKE    323
REAL     77
Name: count, dtype: int64


In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index().rename(columns={"index": "video"})
df["groupe"] = df["original"].fillna(df["video"])   # un original et ses fakes = même groupe

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
df["fold"] = -1
for i, (_, idx) in enumerate(sgkf.split(df, df["label"], df["groupe"])):
    df.loc[df.index[idx], "fold"] = i

df["split_perso"] = df["fold"].map(lambda f: "test" if f == 0 else "val" if f == 1 else "train")

# Contrôle : aucun groupe ne doit apparaître dans deux ensembles
g = df.groupby("groupe")["split_perso"].nunique()
print("groupes présents dans plusieurs splits :", (g > 1).sum())

print(df.groupby(["split_perso", "label"]).size().unstack())

df[["video", "label", "original", "groupe", "split_perso"]].to_csv("/kaggle/working/splits.csv", index=False)

groupes présents dans plusieurs splits : 0
label        FAKE  REAL
split_perso            
test           41    13
train         241    52
val            41    12


In [3]:
all_videos = pd.DataFrame({
    "video":  df["video"],
    "chemin": dossier + "/" + df["video"],
    "source": "dfdc",
    "label":  df["label"].map({"REAL": 0, "FAKE": 1}),   # 1 = généré par IA
    "groupe": "dfdc_" + df["groupe"].str.replace(".mp4", "", regex=False),
    "split":  df["split_perso"],
})
all_videos.to_csv("/kaggle/working/all_videos.csv", index=False)
print(all_videos.shape)
print(all_videos.head())
print(all_videos.groupby(["split", "label"]).size().unstack())

(400, 6)
            video                                             chemin source  \
0  aagfhgtpmv.mp4  /kaggle/input/competitions/deepfake-detection-...   dfdc   
1  aapnvogymq.mp4  /kaggle/input/competitions/deepfake-detection-...   dfdc   
2  abarnvbtwb.mp4  /kaggle/input/competitions/deepfake-detection-...   dfdc   
3  abofeumbvv.mp4  /kaggle/input/competitions/deepfake-detection-...   dfdc   
4  abqwwspghj.mp4  /kaggle/input/competitions/deepfake-detection-...   dfdc   

   label           groupe  split  
0      1  dfdc_vudstovrck  train  
1      1  dfdc_jdubbvfswz   test  
2      0  dfdc_abarnvbtwb   test  
3      1  dfdc_atvmxvwyns  train  
4      1  dfdc_qzimuostzz    val  
label   0    1
split         
test   13   41
train  52  241
val    12   41


In [4]:
import os
from collections import Counter

racine = "/kaggle/input"
ff = [d for d in os.listdir(racine) if "face" in d.lower() or "ff" in d.lower()]
print("dossiers FF++ trouvés :", ff)

for d in ff:
    base = os.path.join(racine, d)
    ext, taille = Counter(), 0
    for dp, dn, fn in os.walk(base):
        for f in fn:
            ext[os.path.splitext(f)[1].lower()] += 1
            taille += os.path.getsize(os.path.join(dp, f))
    print("\n===", d)
    print("extensions :", dict(ext))
    print("taille totale : %.1f Go" % (taille / 1e9))
    for dp, dn, fn in os.walk(base):
        niveau = dp.replace(base, "").count(os.sep)
        if niveau <= 3:
            print("  " * niveau + os.path.basename(dp) + "/", f"({len(fn)} fichiers)")

dossiers FF++ trouvés : []


In [5]:
import os
racine = "/kaggle/input"
for dp, dn, fn in os.walk(racine):
    niveau = dp.replace(racine, "").count(os.sep)
    if niveau <= 3:
        print("  " * niveau + os.path.basename(dp) + "/", f"({len(fn)} fichiers)")
    if niveau > 3:
        dn[:] = []   # ne pas descendre plus profond

input/ (0 fichiers)
  notebooks/ (0 fichiers)
    anonnosingharay/ (0 fichiers)
    robikscube/ (0 fichiers)
  competitions/ (0 fichiers)
    deepfake-detection-challenge/ (1 fichiers)
      test_videos/ (400 fichiers)
      train_sample_videos/ (401 fichiers)
  datasets/ (0 fichiers)
    hungle3401/ (0 fichiers)
      faceforensics/ (0 fichiers)


In [6]:
import os
from collections import Counter

base = "/kaggle/input/datasets/hungle3401/faceforensics"

ext, taille, nb = Counter(), 0, 0
for dp, dn, fn in os.walk(base):
    for f in fn:
        ext[os.path.splitext(f)[1].lower()] += 1
        taille += os.path.getsize(os.path.join(dp, f))
        nb += 1
print("fichiers :", nb)
print("extensions :", dict(ext.most_common(10)))
print("taille totale : %.1f Go" % (taille / 1e9))

print("\nstructure :")
for dp, dn, fn in os.walk(base):
    niveau = dp.replace(base, "").count(os.sep)
    if niveau <= 3:
        print("  " * niveau + os.path.basename(dp) + "/", f"({len(fn)} fichiers)")
    else:
        dn[:] = []

fichiers : 400
extensions : {'.mp4': 400}
taille totale : 2.9 Go

structure :
faceforensics/ (0 fichiers)
  FF++/ (0 fichiers)
    fake/ (200 fichiers)
    real/ (200 fichiers)


In [7]:
import os, cv2
base = "/kaggle/input/datasets/hungle3401/faceforensics/FF++"

for cat in ["real", "fake"]:
    noms = sorted(os.listdir(f"{base}/{cat}"))
    print(f"\n{cat} : {len(noms)} fichiers, exemples :", noms[:8])

    # propriétés de 5 vidéos
    for n in noms[:5]:
        cap = cv2.VideoCapture(f"{base}/{cat}/{n}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        w, h = int(cap.get(3)), int(cap.get(4))
        nb = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"  {n}: {w}x{h}, {fps:.0f} fps, {nb} frames ({nb/fps:.1f} s)")
        cap.release()


real : 200 fichiers, exemples : ['01__exit_phone_room.mp4', '01__hugging_happy.mp4', '01__kitchen_pan.mp4', '01__kitchen_still.mp4', '01__meeting_serious.mp4', '01__outside_talking_pan_laughing.mp4', '01__outside_talking_still_laughing.mp4', '01__podium_speech_happy.mp4']
  01__exit_phone_room.mp4: 1920x1080, 24 fps, 305 frames (12.7 s)
  01__hugging_happy.mp4: 1920x1080, 24 fps, 787 frames (32.8 s)
  01__kitchen_pan.mp4: 1920x1080, 24 fps, 560 frames (23.3 s)
  01__kitchen_still.mp4: 1920x1080, 24 fps, 800 frames (33.3 s)
  01__meeting_serious.mp4: 1920x1080, 24 fps, 1045 frames (43.5 s)

fake : 200 fichiers, exemples : ['01_02__outside_talking_still_laughing__YVGY8LOK.mp4', '01_02__walk_down_hall_angry__YVGY8LOK.mp4', '01_03__hugging_happy__ISF9SP4G.mp4', '01_03__podium_speech_happy__480LQD1C.mp4', '01_03__talking_against_wall__JZUXXFRB.mp4', '01_11__meeting_serious__9OM3VE0Y.mp4', '01_11__secret_conversation__4OJNJLOO.mp4', '01_11__talking_against_wall__9229VVZ3.mp4']
  01_02__outs

In [9]:
import os, collections
base = "/kaggle/input/datasets/hungle3401/faceforensics/FF++"
reels = set(os.listdir(f"{base}/real"))
fakes = os.listdir(f"{base}/fake")

acteurs_reels = collections.Counter(n.split("__")[0] for n in reels)
print("acteurs (vidéos réelles) :", len(acteurs_reels), sorted(acteurs_reels))

premier, second, aucun = 0, 0, 0
paires = set()
for n in fakes:
    a, b = n.split("__")[0].split("_")
    scene = n.split("__")[1]
    paires.add((a, b))
    p = f"{a}__{scene}.mp4" in reels
    s = f"{b}__{scene}.mp4" in reels
    premier += p; second += s; aucun += (not p and not s)

print("\nfausses vidéos :", len(fakes))
print("la vidéo réelle du 1er acteur existe :", premier)
print("la vidéo réelle du 2e acteur existe  :", second)
print("aucune réelle correspondante         :", aucun)
print("nombre de paires d'acteurs :", len(paires))

acteurs (vidéos réelles) : 16 ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16']

fausses vidéos : 200
la vidéo réelle du 1er acteur existe : 200
la vidéo réelle du 2e acteur existe  : 124
aucune réelle correspondante         : 0
nombre de paires d'acteurs : 85


In [10]:
import os
import numpy as np
import pandas as pd

base = "/kaggle/input/datasets/hungle3401/faceforensics/FF++"
lignes = []
for n in sorted(os.listdir(f"{base}/real")):
    a = n.split("__")[0]
    lignes.append(dict(video=n, chemin=f"{base}/real/{n}", source="dfd", label=0, groupe=f"dfd_{a}"))
for n in sorted(os.listdir(f"{base}/fake")):
    a = n.split("__")[0].split("_")[0]          # acteur de la vidéo de base
    lignes.append(dict(video=n, chemin=f"{base}/fake/{n}", source="dfd", label=1, groupe=f"dfd_{a}"))
dfd = pd.DataFrame(lignes)

acteurs = sorted(dfd["groupe"].unique())
np.random.RandomState(42).shuffle(acteurs)
test_a, val_a = acteurs[:3], acteurs[3:6]
dfd["split"] = dfd["groupe"].map(lambda g: "test" if g in test_a else "val" if g in val_a else "train")

print(dfd.groupby(["split", "label"]).size().unstack())

tout = pd.concat([all_videos, dfd], ignore_index=True)
print("\nTable commune :", tout.shape)
print(tout.groupby(["source", "split", "label"]).size().unstack())
tout.to_csv("/kaggle/working/all_videos.csv", index=False)

label      0      1
split              
test    46.0   86.0
train  110.0  114.0
val     44.0    NaN

Table commune : (800, 6)
label             0      1
source split              
dfd    test    46.0   86.0
       train  110.0  114.0
       val     44.0    NaN
dfdc   test    13.0   41.0
       train   52.0  241.0
       val     12.0   41.0


In [13]:
import numpy as np, pandas as pd

cnt = dfd.groupby(["groupe", "label"]).size().unstack(fill_value=0)
print(cnt, "\n")

acteurs = list(cnt.index)
tot = cnt.values.sum()
ratio_f = cnt[1].sum() / tot
cibles = {"test": 0.15, "val": 0.15, "train": 0.70}

rng = np.random.RandomState(0)
meilleur = None
for _ in range(20000):
    p = rng.permutation(acteurs)
    ens = {"test": p[:3], "val": p[3:6], "train": p[6:]}
    score, ok = 0, True
    for k, a in ens.items():
        c = cnt.loc[list(a)].sum()
        if c[0] < 5 or c[1] < 5:          # au moins 5 vidéos de chaque classe
            ok = False
            break
        score += abs(c.sum() / tot - cibles[k]) + abs(c[1] / c.sum() - ratio_f)
    if ok and (meilleur is None or score < meilleur[0]):
        meilleur = (score, ens)

ens = meilleur[1]
dfd["split"] = dfd["groupe"].map(
    lambda g: next(k for k, a in ens.items() if g in set(a)))
print(dfd.groupby(["split", "label"]).size().unstack())

dfdc = pd.read_csv("/kaggle/working/all_videos.csv").query("source == 'dfdc'")
tout = pd.concat([dfdc, dfd], ignore_index=True)
print("\n", tout.groupby(["source", "split", "label"]).size().unstack())
tout.to_csv("/kaggle/working/all_videos.csv", index=False)

label    0   1
groupe        
dfd_01  16  16
dfd_02  16  40
dfd_03  16  40
dfd_04  12  16
dfd_05  11   8
dfd_06  14  30
dfd_07  13  27
dfd_08  10   3
dfd_09   9  17
dfd_10  11   3
dfd_11  12   0
dfd_12  15   0
dfd_13  15   0
dfd_14  15   0
dfd_15  14   0
dfd_16   1   0 

label    0    1
split          
test    30   30
train  136  137
val     34   33

 label           0    1
source split          
dfd    test    30   30
       train  136  137
       val     34   33
dfdc   test    13   41
       train   52  241
       val     12   41


In [1]:
import os
print(os.listdir("/kaggle/working"))

[]
